In [378]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

import os
from tqdm import tqdm

from utils.config import device
from itertools import combinations

In [379]:
path_name = "location_ST_1200"

In [380]:
data_x_path = f"final_loaded_data/{path_name}/weighed/train/data_x.npz"
data_y_path = f"final_loaded_data/{path_name}/weighed/train/data_y.npz"

races_x = np.load(data_x_path)
races_y = np.load(data_y_path)

In [381]:
def get_mean_std(data_x):
    race_ids = list(data_x.keys())
    grouped = []
    for race_id in race_ids:
        grouped.append(torch.tensor(data_x[race_id], device="cpu", dtype=torch.float32))
    grouped = torch.cat(grouped, dim=0)
    mean = torch.mean(grouped, dim=0)
    std = torch.std(grouped, dim=0)
    std[std == 0] = 1
    return mean, std

mean, std = get_mean_std(races_x)
mean = mean.to(device)
std = std.to(device)

In [382]:
def transform_one_race(x):
    # x is a tensor
    m = x.size(0)
    pairs = list(combinations(range(m), 2))
    idx_i = [i for i, j in pairs]
    idx_j = [j for i, j in pairs]

    x_i = x[idx_i]
    x_j = x[idx_j]

    pairs = torch.cat([x_i, x_j], dim=1)
    return pairs


def transform_one_race_y(y):
    m = y.size(0)
    pairs = list(combinations(range(m), 2))
    idx_i = [i for i, j in pairs]
    idx_j = [j for i, j in pairs]

    y_i = y[idx_i]
    y_j = y[idx_j]

    pairwise_y = (y_i < y_j).float()

    return pairwise_y


In [383]:
data_x = []
data_y = []
race_ids = list(races_x.keys())
print(f"Number of races: {len(race_ids)}")
for race_id in race_ids:
    this_race = torch.tensor(races_x[race_id], dtype=torch.float32, device=device)
    this_race = (this_race - mean) / std
    this_race_y = torch.tensor(races_y[race_id][:, 0], dtype=torch.float32, device=device)
    data_x.append(transform_one_race(this_race))
    data_y.append(transform_one_race_y(this_race_y))

data_x = torch.cat(data_x, dim=0)
data_y = torch.cat(data_y, dim=0)

print(data_x.size())
print(data_y.size())

Number of races: 1088
torch.Size([61313, 128])
torch.Size([61313])


In [384]:
def shuffle_indices(m, cv_ratio=0.2):
    indices = torch.randperm(m)
    cv_idx = int(m * (1 - cv_ratio))
    return indices[:cv_idx], indices[cv_idx:]

train_idx, cv_idx = shuffle_indices(len(data_x))

In [385]:
train_x = data_x[train_idx]
train_y = data_y[train_idx]
cv_x = data_x[cv_idx]
cv_y = data_y[cv_idx]

In [386]:
class PairwiseBinary(nn.Module):
    def __init__(self):
        super(PairwiseBinary, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(128, 48),
            nn.ReLU(),
            nn.Linear(48, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


In [387]:
model = PairwiseBinary().to(device)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [388]:
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(train_x).flatten()
    loss = criterion(pred, train_y)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        cv_pred = model(cv_x).flatten()
        cv_loss = criterion(cv_pred, cv_y)

    train_acc = torch.mean((torch.round(pred) == train_y).float())
    cv_acc = torch.mean((torch.round(cv_pred) == cv_y).float())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}: loss: {loss.item()}, train acc = {train_acc.item()}, cv loss = {cv_loss.item()}, cv acc = {cv_acc.item()}")


Epoch 100/1000: loss: 0.6845648884773254, train acc = 0.5461570024490356, cv loss = 0.6839929819107056, cv acc = 0.5469298362731934
Epoch 200/1000: loss: 0.6752133965492249, train acc = 0.5716004371643066, cv loss = 0.6743016839027405, cv acc = 0.5731876492500305
Epoch 300/1000: loss: 0.6649333834648132, train acc = 0.6082977056503296, cv loss = 0.6637455224990845, cv acc = 0.6138791441917419
Epoch 400/1000: loss: 0.6534123420715332, train acc = 0.6365341544151306, cv loss = 0.6519051790237427, cv acc = 0.6413602232933044
Epoch 500/1000: loss: 0.6412530541419983, train acc = 0.65217125415802, cv loss = 0.6393630504608154, cv acc = 0.6557938456535339
Epoch 600/1000: loss: 0.6295352578163147, train acc = 0.6597961783409119, cv loss = 0.6272473335266113, cv acc = 0.6618282794952393
Epoch 700/1000: loss: 0.619502604007721, train acc = 0.6649541258811951, cv loss = 0.6168957352638245, cv acc = 0.6661502122879028
Epoch 800/1000: loss: 0.6118781566619873, train acc = 0.6683180928230286, cv lo

In [389]:
def compute_score_ranks(pairwise_scores, m):
    # Infer number of horses from pairwise count
    score_vector = torch.zeros(m, dtype=pairwise_scores.dtype, device=device)

    k = 0
    for i in range(m):
        for j in range(i + 1, m):
            p_ij = pairwise_scores[k]
            score_vector[i] += p_ij      # i gets p_ij
            score_vector[j] += 1 - p_ij  # j gets (1 - p_ij)
            k += 1

    return score_vector


In [390]:
test_x = np.load(f"final_loaded_data/{path_name}/weighed/test/data_x.npz")
test_y = np.load(f"final_loaded_data/{path_name}/weighed/test/data_y.npz")

In [391]:
model.eval()

win = 0
place = 0
total = 0

race_ids = list(test_x.keys())
for race_id in race_ids:
    race_x = test_x[race_id]
    m = len(race_x)

    race_x = (torch.tensor(race_x, dtype=torch.float32, device=device) - mean) / std
    race_x = transform_one_race(race_x)
    race_y = test_y[race_id]
    scores = model(race_x).flatten()
    grouped = compute_score_ranks(scores, m)
    grouped_scores = torch.softmax(grouped, dim=0)
    pred_winner = torch.argmax(grouped_scores).item()
    if race_y[pred_winner, 0] == 1:
        win += 1
    if race_y[pred_winner, 0] <= 3:
        place += 1
    total += 1

print(win / total)
print(place / total)

0.2925531914893617
0.5851063829787234
